# HCR Filters

This notebook demonstrates the HCR ROI filtering pipeline, including:
1. Volume-based filtering
2. Comprehensive filtering (soma + edge + tile overlap)
3. Using the helper methods
4. Visualizing filter results

In [3]:
from aind_hcr_data_loader.hcr_dataset import create_hcr_dataset_from_config, HCRDataset
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

import aind_hcr_data_loader.filters as hcr_filters
import numpy as np

# notebook reload
%load_ext autoreload
%autoreload 2

In [5]:
data_dir = Path("/root/capsule/data")
config_path = "/src/aind-hcr-data-loader/src/aind_hcr_data_loader/examples/mouse_config_example.json"

mouse_id = "767018" # must be in config file
dataset = create_hcr_dataset_from_config(mouse_id,
                                         data_dir=data_dir,
                                         config_path=config_path)
print(dataset)

mouse_id: 767018
HCRDataset(mouse_id='767018', rounds=['R1', 'R2', 'R3', 'R4', 'R5'], total_channels=28)


## Method 1: Using the Helper Method (Recommended)

The easiest way to get filtered cell IDs is using the `get_filtered_cell_ids()` helper method on the dataset object.

In [ ]:
# Get comprehensive filtered cell IDs (default settings)
filtered_ids = dataset.get_filtered_cell_ids(
    filter_type='comprehensive',
    verbose=True
)

print(f"\nFiltered cell IDs: {len(filtered_ids):,}")

Loading mixed cxg for round R1
Number of cells in mixed_cxg for round R1: 93062
Loading mixed cxg for round R2
Number of cells in mixed_cxg for round R2: 90238
Loading mixed cxg for round R3
Number of cells in mixed_cxg for round R3: 87079
Loading mixed cxg for round R4
Number of cells in mixed_cxg for round R4: 88693
Loading mixed cxg for round R5
Number of cells in mixed_cxg for round R5: 82150
ROI FILTERING PIPELINE

[1/5] Loading metrics from: /root/capsule/scratch/shape_metrics/HCR_767018_2025-08-20_14-00-00_processed_2025-09-10_00-33-24/seg_shape_metrics_pyr2.parquet
      Total ROIs loaded: 101,494

[2/5] Applying volume filter...
Filtered out by volume (10-98 percentile): 89,328
      Kept: 89,328 ROIs
      Filtered out: 12,166 ROIs (volume outliers)

[3/5] Running soma classifier (threshold=0.8)...
----------------------------------------
✓ Loaded classifier from simple_soma_1.joblib
Inference complete on 89,328 rois
Predicted soma: 81,873 (91.65%)
Predicted non-soma: 7,455 (

/src/aind-hcr-data-loader/src/aind_hcr_data_loader/filters.py:661: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  df_out._boundary_info = {
/src/aind-hcr-data-loader/src/aind_hcr_data_loader/filters.py:666: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  df_out._thresholds = {
/src/aind-hcr-qc/src/aind_hcr_qc/viz/tile_alignment.py:4649: RuntimeWarning: invalid value encountered in divide
  overlap_fractions = intersect_areas / roi_areas[potential_intersect_mask]


In [7]:
# Get comprehensive filtered cell IDs with custom thresholds
filtered_ids_custom = dataset.get_filtered_cell_ids(
    filter_type='comprehensive',
    soma_threshold=0.85,        # Stricter soma classification
    edge_xy_threshold=15.0,     # Larger XY edge exclusion zone
    edge_z_threshold=150.0,     # Larger Z edge exclusion zone
    overlap_threshold=0.15,     # More conservative tile overlap filtering
    verbose=True
)

print(f"\nWith custom thresholds: {len(filtered_ids_custom):,} cells")

Loading mixed cxg for round R1
Loading mixed cxg for round R2
Number of cells in mixed_cxg for round R2: 90238
Loading mixed cxg for round R3
Number of cells in mixed_cxg for round R3: 87079
Loading mixed cxg for round R4
Number of cells in mixed_cxg for round R4: 88693
Loading mixed cxg for round R5
Number of cells in mixed_cxg for round R5: 82150
ROI FILTERING PIPELINE

[1/5] Loading metrics from: /root/capsule/scratch/shape_metrics/HCR_767018_2025-08-20_14-00-00_processed_2025-09-10_00-33-24/seg_shape_metrics_pyr2.parquet
      Total ROIs loaded: 101,494

[2/5] Applying volume filter...
Filtered out by volume (10-98 percentile): 89,328
      Kept: 89,328 ROIs
      Filtered out: 12,166 ROIs (volume outliers)

[3/5] Running soma classifier (threshold=0.85)...
----------------------------------------
✓ Loaded classifier from simple_soma_1.joblib
Inference complete on 89,328 rois
Predicted soma: 80,982 (90.66%)
Predicted non-soma: 8,346 (9.34%)
----------------------------------------


/src/aind-hcr-data-loader/src/aind_hcr_data_loader/filters.py:661: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  df_out._boundary_info = {
/src/aind-hcr-data-loader/src/aind_hcr_data_loader/filters.py:666: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  df_out._thresholds = {


      Core ROIs: 61,686
      Edge ROIs: 27,642 (to filter)

[5/5] Running tile boundary overlap filter...
Dataset path: HCR_767018_2025-08-20_14-00-00_processed_2025-09-10_00-33-24/corrected.ome.zarr/
405: n=25
Found 40 adjacent tile pairs
Loading metrics from /root/capsule/scratch/shape_metrics/HCR_767018_2025-08-20_14-00-00_processed_2025-09-10_00-33-24/seg_shape_metrics_pyr2.parquet
Global offset: x_min=-3654.9, y_min=-3691.3
Processing 101494 ROIs against 40 overlap regions
Filtered 12449 / 101494 ROIs (12.3%)
Remaining: 89045 ROIs
      ROIs in tile overlaps: 12,449 (to filter)

COMBINING FILTERS

Filter breakdown:
  Non-soma:           8,346
  Edge ROIs:         27,642
  Tile overlaps:     12,449
  ------------------------------
  Total unique:      39,329 (filtered out)
  Kept:              62,165 (61.2%)


Comprehensive filtering: keeping 60574/97915 cells

With custom thresholds: 60,574 cells


/src/aind-hcr-qc/src/aind_hcr_qc/viz/tile_alignment.py:4649: RuntimeWarning: invalid value encountered in divide
  overlap_fractions = intersect_areas / roi_areas[potential_intersect_mask]


In [ ]:
# Get volume filtered cell IDs (simpler, faster)
volume_filtered_ids = dataset.get_filtered_cell_ids(
    filter_type='volume',
    verbose=True
)

print(f"\nVolume filtered: {len(volume_filtered_ids):,} cells")

## Method 2: Using roi_filter_comprehensive Directly

For advanced use cases, you can call the comprehensive filter function directly to get detailed results.

In [8]:
# Call comprehensive filter directly to get detailed results
results = hcr_filters.roi_filter_comprehensive(
    dataset,
    round_key='R1',
    soma_threshold=0.8,
    edge_xy_threshold=10.0,
    edge_z_threshold=10.0,
    overlap_threshold=0.1,
    verbose=True
)

# Results is a dictionary with detailed information
print("\nResults keys:", results.keys())
print(f"\nTotal ROIs: {results['n_total']:,}")
print(f"Filtered out: {results['n_filtered']:,}")
print(f"Kept: {results['n_kept']:,}")

ROI FILTERING PIPELINE

[1/5] Loading metrics from: /root/capsule/scratch/shape_metrics/HCR_767018_2025-08-20_14-00-00_processed_2025-09-10_00-33-24/seg_shape_metrics_pyr2.parquet
      Total ROIs loaded: 101,494

[2/5] Applying volume filter...
Filtered out by volume (10-98 percentile): 89,328
      Kept: 89,328 ROIs
      Filtered out: 12,166 ROIs (volume outliers)

[3/5] Running soma classifier (threshold=0.8)...
----------------------------------------
✓ Loaded classifier from simple_soma_1.joblib
Inference complete on 89,328 rois
Predicted soma: 81,873 (91.65%)
Predicted non-soma: 7,455 (8.35%)
----------------------------------------
      Soma ROIs: 81,873
      Non-soma ROIs: 7,455 (to filter)

[4/5] Running edge ROI classifier...
      XY threshold: 10.0 pixels
      Z threshold: 10.0 pixels
      Core ROIs: 76,039
      Edge ROIs: 13,289 (to filter)

[5/5] Running tile boundary overlap filter...
Dataset path: HCR_767018_2025-08-20_14-00-00_processed_2025-09-10_00-33-24/correc

/src/aind-hcr-data-loader/src/aind_hcr_data_loader/filters.py:661: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  df_out._boundary_info = {
/src/aind-hcr-data-loader/src/aind_hcr_data_loader/filters.py:666: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  df_out._thresholds = {
/src/aind-hcr-qc/src/aind_hcr_qc/viz/tile_alignment.py:4649: RuntimeWarning: invalid value encountered in divide
  overlap_fractions = intersect_areas / roi_areas[potential_intersect_mask]


In [10]:
# Access individual filter results
print("Filter breakdown:")
print(f"  Non-soma IDs: {len(results['soma_filter_ids']):,}")
print(f"  Edge ROI IDs: {len(results['edge_filter_ids']):,}")
print(f"  Tile overlap IDs: {len(results['overlap_filter_ids']):,}")

Filter breakdown:
  Non-soma IDs: 7,455
  Edge ROI IDs: 13,289
  Tile overlap IDs: 13,174


## Method 3: Load Spots with Automatic Filtering

You can directly load spots with filtering applied automatically.

In [ ]:
# Load all spots with comprehensive filtering (parallelized)
spots_comprehensive = dataset.load_all_rounds_spots_mp(
    table_type='mixed_spots',
    roi_filter_type='comprehensive'
)

print(f"Loaded spots with comprehensive filtering: {len(spots_comprehensive):,}")
print(f"Unique cells: {spots_comprehensive['cell_id'].nunique():,}")
print(f"\nColumns: {list(spots_comprehensive.columns)}")

In [ ]:
# Load spots with volume filtering only (faster)
spots_volume = dataset.load_all_rounds_spots_mp(
    table_type='mixed_spots',
    roi_filter_type='volume'
)

print(f"Loaded spots with volume filtering: {len(spots_volume):,}")
print(f"Unique cells: {spots_volume['cell_id'].nunique():,}")

In [ ]:
# Create cell×gene matrix with comprehensive filtering
cxg, spots = dataset.create_cell_gene_matrix_with_spots(
    table_type='mixed_spots',
    roi_filter_type='comprehensive',
    return_spots=True
)

print(f"Cell×gene matrix shape: {cxg.shape}")
print(f"  Cells: {cxg.shape[0]:,}")
print(f"  Genes: {cxg.shape[1]:,}")
print(f"\nSpots dataframe: {len(spots):,} spots")

## Method 4: Using Edge ROI Classifier Directly

For visualizing and understanding edge filtering, you can use the EdgeROIClassifier directly.

In [ ]:
# Load metrics data for edge classification
dataset_name = dataset.rounds['R1'].name
metrics_path = Path(f"/root/capsule/scratch/shape_metrics/{dataset_name}/seg_shape_metrics_pyr2.parquet")

if metrics_path.exists():
    metrics_df = pd.read_parquet(metrics_path)
    
    # Create edge classifier
    edge_classifier = hcr_filters.EdgeROIClassifier(
        xy_distance_threshold=10.0,
        z_distance_threshold=100.0
    )
    
    # Classify ROIs
    classified_df = edge_classifier.classify(metrics_df, verbose=True)
    
    print(f"\nClassified {len(classified_df):,} ROIs")
    print(f"Edge ROIs: {(classified_df['is_edge_roi']).sum():,}")
    print(f"Core ROIs: {(~classified_df['is_edge_roi']).sum():,}")
else:
    print(f"Metrics file not found: {metrics_path}")

In [ ]:
# Visualize edge classification (if metrics were loaded)
if 'classified_df' in locals():
    # Plot for different orientations
    for orientation in ['ZX', 'ZY', 'XY']:
        fig, axes = edge_classifier.plot(
            classified_df, 
            orientation=orientation,
            figsize=(18, 5),
            save=False  # Set to True to save figures
        )
        plt.show()

## Summary: Filter Types

### Filter Type Comparison

| Filter Type | Speed | Removes | Use Case |
|------------|-------|---------|----------|
| `'none'` | Instant | Nothing | No filtering, all cells |
| `'volume'` | Fast | Volume outliers (q<0.2, q>0.95) | Quick quality check |
| `'comprehensive'` | Slower | Non-soma + Edge + Tile overlap | Production analysis |

### Available Parameters for 'comprehensive'

- `soma_threshold`: Confidence for soma classification (default: 0.8)
- `edge_xy_threshold`: XY edge distance in pixels (default: 10.0)
- `edge_z_threshold`: Z edge distance in pixels (default: 100.0)
- `overlap_threshold`: Tile overlap fraction (default: 0.1)
- `metrics_base_path`: Path to metrics files
- `round_key`: Round to use for metrics (default: 'R1')
- `verbose`: Show detailed output (default: True)